# 04 · Synthetic training data and the full fine-tune that failed

First attempt at adapting `mxbai-embed-large-v1` to movie search:

1. generate ~5 K synthetic search queries for random movies with an LLM;
2. fine-tune **all** weights of the embedding model on (query, plot) pairs.

**Outcome: the model collapsed.** Hit@1 on the 394-query test set fell from 37.3 % (base model) to 11.2 %, and even the easy `oracle` queries dropped from 74 % to 18 %. With every parameter trainable, a tiny batch, and only synthetic queries as supervision, the model overfit the query style and lost its general retrieval ability.

The fix was to freeze the base model and train a small LoRA adapter instead (notebooks 05 → 06). This notebook is kept as the record of what was tried and why it was abandoned.

In [1]:
import pandas as pd
import time
import json
import re


c:\Users\MEDIA\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\MEDIA\AppData\Local\Temp\ipykernel_32572\4032879178.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## Step 1 · Synthetic queries with Llama-3.1-8B (Groq)

Two queries per movie: a keyword-style *naturalistic* query and a *conversational* one ("that movie where..."). Requires `GROQ_API_KEY` in the environment; the loop is resumable.

In [ ]:
import os
from groq import Groq

GROQ_API_KEY = os.environ["GROQ_API_KEY"]
INPUT_CSV    = "../data/cleaned_movie_plots_v2.csv"
OUTPUT_CSV   = "../data/training/synthetic_training_pairs.csv"
DELAY        = 1.5

client = Groq(api_key=GROQ_API_KEY)

In [ ]:
import os

df_all = pd.read_csv(INPUT_CSV).dropna(subset=["title", "release_year"])

if os.path.exists(OUTPUT_CSV):
    df_done     = pd.read_csv(OUTPUT_CSV)
    done_titles = set(df_done["title"].unique())
    current_count = len(df_done)
    print(f"Resuming — {len(done_titles)} movies done, {current_count} queries saved so far.")
else:
    done_titles   = set()
    current_count = 0
    pd.DataFrame(columns=["title", "year", "genre", "query", "query_type"]).to_csv(OUTPUT_CSV, index=False)
    print("Starting fresh.")

TARGET = 6000

PROMPT_TEMPLATE = """You are helping build a movie search dataset.
Given the movie title, release year, and plot below, generate exactly 2 search queries a real user might type to find this movie.

Query 1 (naturalistic): short keyword-style query based on the plot. No filler words. Focus on the main theme, characters, or events.
Query 2 (conversational): casual phrasing using filler words like "that movie where..." or "the one where...". Write it like a person trying to remember the movie from memory.

IMPORTANT: Do NOT just repeat the title. Base the queries on what actually happens in the plot.

Return ONLY a valid JSON object in this exact format, nothing else, no markdown:
{{"naturalistic": "...", "conversational": "..."}}

Movie title: {title}
Release year: {year}
Plot: {plot}"""

df_remaining = df_all[~df_all["title"].isin(done_titles)].sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Titles remaining: {len(df_remaining)}")
print(f"Queries needed: {TARGET - current_count}\n")

for _, row in df_remaining.iterrows():

    if current_count >= TARGET:
        print(f"Target of {TARGET} queries reached. Done!")
        break

    title = str(row["title"]).strip()
    year  = str(row["release_year"]).strip()
    genre = str(row.get("genre", "unknown")).strip()
    plot  = str(row.get("plot", "")).strip()[:500]

    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": PROMPT_TEMPLATE.format(title=title, year=year, plot=plot)}],
            temperature=0.7,
            response_format={"type": "json_object"},
        )

        text    = response.choices[0].message.content.strip()
        queries = json.loads(text)

        new_rows = pd.DataFrame([
            {"title": title, "year": year, "genre": genre, "query": queries["naturalistic"],   "query_type": "naturalistic"},
            {"title": title, "year": year, "genre": genre, "query": queries["conversational"], "query_type": "conversational"},
        ])
        new_rows.to_csv(OUTPUT_CSV, mode="a", header=False, index=False)

        current_count += 2
        done_titles.add(title)
        print(f"[{current_count}/{TARGET}] ✓ {title} ({year})")

    except Exception as e:
        error_msg = str(e)
        if "429" in error_msg:
            print(f"Rate limit hit — waiting 30 seconds...")
            time.sleep(30)
        else:
            print(f"✗ {title} ({year}) — {e}")

    time.sleep(DELAY)

print(f"\nFinal count: {current_count} queries saved to {OUTPUT_CSV}")

## Step 2 · Attach plots and filter short queries

Queries shorter than five words are dropped (they are mostly title fragments).

In [2]:
df_plots = pd.read_csv("../data/cleaned_movie_plots_v2.csv")
df_train = pd.read_csv("../data/training/synthetic_training_pairs.csv")

df_merged = df_train.merge(df_plots[["title", "plot"]], on="title", how="left")
df_merged = df_merged.dropna(subset=["plot"])
print(f"Training pairs with plots: {len(df_merged)}")
df_merged.to_csv("../data/training/training_pairs_with_plots.csv", index=False)

Training pairs with plots: 5202


In [4]:
import pandas as pd

df = pd.read_csv("../data/training/training_pairs_with_plots.csv")
print(f"Before filtering: {len(df)}")

# filter out short queries
df['query_len'] = df['query'].str.split().str.len()
df = df[df['query_len'] >= 5].reset_index(drop=True)
print(f"After removing short queries: {len(df)}")

print(df['query_type'].value_counts())
print(df['query_len'].describe())

df.to_csv("../data/training/training_pairs_clean.csv", index=False)
print("\nSaved to training_pairs_clean.csv")

Before filtering: 5202
After removing short queries: 3250
query_type
conversational    2601
naturalistic       649
Name: count, dtype: int64
count    3250.000000
mean       12.673231
std         4.724316
min         5.000000
25%        10.000000
50%        13.000000
75%        16.000000
max        33.000000
Name: query_len, dtype: float64

Saved to training_pairs_clean.csv


## Step 3 · Full fine-tuning runs (all 335 M parameters)

Two runs of `MultipleNegativesRankingLoss` on CPU with batch size 4, i.e. only three in-batch negatives per step. The first cell is the later run (1 epoch on the filtered pairs); the second is the original 3-epoch run on all pairs. Plots are truncated to 256 characters to fit into memory.

In [6]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import torch
import math
import pandas as pd
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

TRAIN_CSV    = "../data/training/training_pairs_clean.csv"
OUTPUT_PATH  = "../models/full_finetune_v2"
BASE_MODEL   = "mixedbread-ai/mxbai-embed-large-v1"
BATCH_SIZE   = 4
EPOCHS       = 1
WARMUP_RATIO = 0.1
MAX_SEQ_LEN  = 128
PLOT_CHARS   = 256

print(f"CPU threads: {torch.get_num_threads()}")

df = pd.read_csv(TRAIN_CSV)
print(f"Training examples: {len(df)}")

examples = [
    InputExample(texts=[
        f"Represent this sentence for searching relevant passages: {row['query']}",
        str(row["plot"]).strip()[:PLOT_CHARS]
    ])
    for _, row in df.iterrows()
]

model = SentenceTransformer(BASE_MODEL)
model.max_seq_length = MAX_SEQ_LEN
model[0].auto_model.gradient_checkpointing_enable()

print(f"Model loaded. Max seq length: {model.max_seq_length}")

dataloader = DataLoader(
    examples,
    shuffle=True,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

loss = losses.MultipleNegativesRankingLoss(model)

warmup_steps = math.ceil(len(dataloader) * EPOCHS * WARMUP_RATIO)
total_steps  = len(dataloader) * EPOCHS
print(f"Total steps: {total_steps} | Warmup steps: {warmup_steps}")
print(f"Estimated time: {total_steps * 1.5 / 60:.0f}–{total_steps * 3 / 60:.0f} minutes on CPU\n")

model.fit(
    train_objectives=[(dataloader, loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    output_path=OUTPUT_PATH,
    show_progress_bar=True,
    checkpoint_path=OUTPUT_PATH + "/checkpoints",
    checkpoint_save_steps=500,
)

print(f"\nDone. Model saved to {OUTPUT_PATH}")

CPU threads: 10
Training examples: 3250
Model loaded. Max seq length: 128
Total steps: 813 | Warmup steps: 82
Estimated time: 20–41 minutes on CPU



Step,Training Loss
500,0.118200



Done. Model saved to ../models/finetuned_movie_model_v2


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # must be before any torch import

import torch
import math
import pandas as pd
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

# ── config ───────────────────────────────────────────────────────────────────
TRAIN_CSV    = "../data/training/training_pairs_with_plots.csv"
OUTPUT_PATH  = "../models/full_finetune_v1"
BASE_MODEL   = "mixedbread-ai/mxbai-embed-large-v1"
BATCH_SIZE   = 4
EPOCHS       = 3
WARMUP_RATIO = 0.1
MAX_SEQ_LEN  = 128
PLOT_CHARS   = 256
# ─────────────────────────────────────────────────────────────────────────────

print(f"CPU threads: {torch.get_num_threads()}")

# load training data
df = pd.read_csv(TRAIN_CSV)
print(f"Training examples: {len(df)}")

# build input pairs
examples = [
    InputExample(texts=[
        f"Represent this sentence for searching relevant passages: {row['query']}",
        str(row["plot"]).strip()[:PLOT_CHARS]
    ])
    for _, row in df.iterrows()
]

# load model
model = SentenceTransformer(BASE_MODEL)

# reduce sequence length to save memory
model.max_seq_length = MAX_SEQ_LEN

# enable gradient checkpointing — trades speed for memory
model[0].auto_model.gradient_checkpointing_enable()

print(f"Model loaded. Max seq length: {model.max_seq_length}")

# dataloader
dataloader = DataLoader(
    examples,
    shuffle=True,
    batch_size=BATCH_SIZE,
    num_workers=0,
)

# loss
loss = losses.MultipleNegativesRankingLoss(model)

warmup_steps = math.ceil(len(dataloader) * EPOCHS * WARMUP_RATIO)
total_steps  = len(dataloader) * EPOCHS
print(f"Total steps: {total_steps} | Warmup steps: {warmup_steps}")
print(f"Estimated time: {total_steps * 1.5 / 60:.0f}–{total_steps * 3 / 60:.0f} minutes on CPU\n")

# fine-tune
model.fit(
    train_objectives=[(dataloader, loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    output_path=OUTPUT_PATH,
    show_progress_bar=True,
    checkpoint_path=OUTPUT_PATH + "/checkpoints",
    checkpoint_save_steps=500,
)

print(f"\nDone. Model saved to {OUTPUT_PATH}")

c:\Users\MEDIA\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CPU threads: 10
Training examples: 5202
Model loaded. Max seq length: 128
Total steps: 3903 | Warmup steps: 391
Estimated time: 98–195 minutes on CPU



Step,Training Loss
500,0.137500
1000,0.105200
1500,0.075300
2000,0.033400
2500,0.027900
3000,0.010500
3500,0.008200



Done. Model saved to ../models/finetuned_movie_model


## Step 4 · Re-embed the corpus with the fine-tuned model

Evaluated with the same procedure as notebook 07. Result on the 394-query test set: **Hit@1 11.2 %, Hit@10 37.8 %, MRR 0.172** versus 37.3 % / 52.0 % / 0.415 for the untouched base model (`results/eval_394_full_finetune.csv`). The fine-tune made the model much worse; see the header above.

In [8]:
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer
import pandas as pd
import faiss

# load fine-tuned model instead of the original
model = SentenceTransformer("../models/full_finetune_v2")

# load your dataset
csv_path = "../data/cleaned_movie_plots_v2.csv"
df = pd.read_csv(csv_path)

# re-embed all plots
def build_embedding_text(row):
    return str(row['plot']).strip()

texts = df.apply(build_embedding_text, axis=1).tolist()

embeddings = model.encode(
    texts,
    batch_size=16,   # start smaller because this model is much larger
    show_progress_bar=True,
    normalize_embeddings=True
)

# normalize
faiss.normalize_L2(embeddings)

# save new embeddings and metadata
np.save("../artifacts/legacy/embeddings_full_finetune.npy", embeddings)

metadata = df.to_dict(orient="records")
with open("../artifacts/legacy/metadata_full_finetune.pkl", "wb") as f:
    pickle.dump(metadata, f)

print("Done. New embeddings saved.")

<>:11: SyntaxWarning: invalid escape sequence '\d'
<>:11: SyntaxWarning: invalid escape sequence '\d'
C:\Users\MEDIA\AppData\Local\Temp\ipykernel_24824\307962147.py:11: SyntaxWarning: invalid escape sequence '\d'
  csv_path = "..\datasets\cleaned_movie_plots_v2.csv"
Batches: 100%|██████████| 2023/2023 [2:49:38<00:00,  5.03s/it]  


Done. New embeddings saved.
